In [62]:
import sqlite3
import pandas as pd

DB_PATH = "/Users/darraghdonnelly/dev/Database/recovered.db"
TEST_SIZE = 0.20
RANDOM_STATE = 42

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        "SELECT src, sex, age, distance_m, time_s FROM concat_results",
        conn,
    )

df.head()


,src,sex,age,distance_m,time_s
0,CherryBlossom2017,1,21,16093,3217.0
1,CherryBlossom2017,1,22,16093,3232.0
2,CherryBlossom2017,1,31,16093,3276.0
3,CherryBlossom2017,1,33,16093,3285.0
4,CherryBlossom2017,1,35,16093,3288.0


In [63]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate

# use same preprocessing steps for all models

df["distance_bucket"] = (
    df["distance_m"].round().astype(int)
    .map({42195: "42k", 16093: "16k", 5000: "5k"})
)
# transform dist to log(dist)
df["log_distance"] = np.log(df["distance_m"].astype(float))

feature_cols = ["log_distance", "age", "sex"]

# train test split stratified by distance
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["distance_bucket"],
)

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

# transform time to log(time)
y_train = np.log(train_df["time_s"].astype(float))
y_test = np.log(test_df["time_s"].astype(float))

# split training by distance
train_buckets = train_df["distance_bucket"]


In [64]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer

# return metrics in seconds intead of log scale
def mae_seconds(y_true_log, y_pred_log):
    return mean_absolute_error(np.exp(y_true_log), np.exp(y_pred_log))

def rmse_seconds(y_true_log, y_pred_log):
    return np.sqrt(mean_squared_error(np.exp(y_true_log), np.exp(y_pred_log)))

def r2_seconds(y_true_log, y_pred_log):
    return r2_score(np.exp(y_true_log), np.exp(y_pred_log))

# evaluation metrics for cv
scoring = {
    "mae": make_scorer(mae_seconds, greater_is_better=False),
    "rmse": make_scorer(rmse_seconds, greater_is_better=False),
    "r2": make_scorer(r2_seconds),
}

# stratified k fold for three distance buckets
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


In [65]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

# dict of models for evaluation 
models = {
    "Mean Baseline": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=0.1),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=3,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        loss="absolute_error",
        learning_rate=0.3,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=100,
        random_state=RANDOM_STATE
    )
}

In [83]:
from sklearn.base import clone

# arrays to store results
rows = []
bucket_rows = []

# loop through models (name = model name, estimator = actual model)
for name, estimator in models.items():

    # run cv on training data
    cv = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=skf.split(X_train, train_buckets),
        scoring=scoring,
        n_jobs=-1,
    )

    # make clean model for test evaluation
    fresh_model = clone(estimator)
    fresh_model.fit(X_train, y_train)
    pred_log = fresh_model.predict(X_test)

    # convert back to seconds
    actual_seconds = np.exp(y_test)
    pred_seconds = np.exp(pred_log)

    # compute test metrics
    test_mae = mean_absolute_error(actual_seconds, pred_seconds)
    test_rmse = np.sqrt(mean_squared_error(actual_seconds, pred_seconds))
    test_r2 = r2_score(actual_seconds, pred_seconds)

    # store results in array
    rows.append({
        "model": name,
        "cv_mae_m": round(-cv["test_mae"].mean() / 60, 2),
        "cv_rmse_m": round(-cv["test_rmse"].mean() / 60, 2),
        "cv_r2": round(cv["test_r2"].mean(), 4),
        "test_rmse_m": round(test_rmse / 60, 2),
        "test_r2": round(test_r2, 4),
    })

    # df for test evaluation by distance bucket
    test_eval = test_df[["distance_bucket"]].copy()
    test_eval["actual_time_s"] = actual_seconds.values
    test_eval["pred_time_s"] = pred_seconds


    for bucket in ["5k", "16k", "42k"]:

        # compute MAE for each distance bucket using test_eval df
        bucket_data = test_eval[test_eval["distance_bucket"] == bucket]
        bucket_mae_s = mean_absolute_error(bucket_data["actual_time_s"], bucket_data["pred_time_s"])

        # store bucket results in different array (convert time to mins)
        bucket_rows.append({
            "model": name,
            "bucket": bucket,
            "test_mae_min": round(bucket_mae_s / 60, 2),
        })

# store results in dfs 
all_results = pd.DataFrame(rows)
bucket_results = pd.DataFrame(bucket_rows)

print("Overall results")
display(all_results.sort_values("cv_mae_m"))

print("Bucketed test MAE (minutes)")
display(bucket_results)


Overall results


,model,cv_mae_m,cv_rmse_m,cv_r2,test_rmse_m,test_r2
4,Gradient Boosting,36.20,50.79,0.7262,50.61,0.7256
5,XGBoost,36.24,50.58,0.7284,50.46,0.7272
1,Linear Regression,36.62,51.30,0.7207,51.10,0.7202
2,Ridge,36.62,51.30,0.7207,51.10,0.7202
3,Random Forest,36.69,51.35,0.7201,51.19,0.7192
0,Mean Baseline,85.31,100.49,-0.0720,99.84,-0.0680


Bucketed test MAE (minutes)


,model,bucket,test_mae_min
0,Mean Baseline,5k,162.91
1,Mean Baseline,16k,106.79
2,Mean Baseline,42k,75.58
3,Linear Regression,5k,10.22
4,Linear Regression,16k,12.74
5,Linear Regression,42k,44.99
6,Ridge,5k,10.22
7,Ridge,16k,12.74
8,Ridge,42k,44.99
9,Random Forest,5k,7.63
